# Práctica 1 — Proyecto GPU: EDA, modelado y benchmarking

Plantilla base para construir una entrega con evidencia verificable de ejecución en GPU NVIDIA.

## Ruta obligatoria

1. Ejecuta el preflight y conserva su salida.
2. Prepara los mismos datos para la ruta CPU y la ruta GPU.
3. Realiza EDA con cuDF y entrena con cuML.
4. Compara CPU/GPU con warmup y sincronización.
5. Exporta artefactos y explica los resultados.

La ruta CPU es el **baseline**. No constituye por sí sola una solución evaluable de esta práctica.

In [ ]:
# 0) Preflight obligatorio: esta celda debe completarse sin errores
import platform
import subprocess
import sys

try:
    gpu_info = subprocess.run(
        [
            "nvidia-smi",
            "--query-gpu=name,driver_version,memory.total",
            "--format=csv,noheader",
        ],
        check=True,
        capture_output=True,
        text=True,
    ).stdout.strip()
except (FileNotFoundError, subprocess.CalledProcessError) as exc:
    raise RuntimeError(
        "No se detecta una GPU NVIDIA operativa. Activa un runtime GPU en "
        "Google Colab o usa el runtime cloud indicado por el profesorado."
    ) from exc

try:
    import cupy as cp
    import cudf
    import cuml
except ImportError as exc:
    raise RuntimeError(
        "Faltan las librerías RAPIDS del runtime: cudf, cuml o su backend CUDA."
    ) from exc

device_id = cp.cuda.Device().id
device_name = cp.cuda.runtime.getDeviceProperties(device_id)["name"]
if isinstance(device_name, bytes):
    device_name = device_name.decode()

# Una operación real confirma que cuDF puede usar el backend CUDA.
_ = cudf.Series([1, 2, 3]).sum()
cp.cuda.Stream.null.synchronize()

print("Python:", sys.version.split()[0], "| Plataforma:", platform.platform())
print("nvidia-smi:", gpu_info)
print("cuDF:", cudf.__version__, "| cuML:", cuml.__version__)
print("Backend: CUDA/RAPIDS | Dispositivo:", device_id, device_name)
print("✅ Preflight GPU superado; conserva esta salida en la entrega.")


In [ ]:
# 1) Datos: sustituye este ejemplo por el dataset de tu proyecto
from pathlib import Path
import json
import time

import numpy as np
import pandas as pd

DATA_PATH = Path("data/dataset.csv")
FEATURES = ["feature1", "feature2"]
TARGET = "target"

if DATA_PATH.exists():
    df_cpu = pd.read_csv(DATA_PATH)
else:
    rng = np.random.default_rng(0)
    rows = 50_000
    feature1 = rng.normal(0, 1, rows)
    feature2 = rng.uniform(-1, 1, rows)
    target = (feature1 + 0.8 * feature2 + rng.normal(0, 0.7, rows) > 0).astype("int32")
    df_cpu = pd.DataFrame({"feature1": feature1, "feature2": feature2, "target": target})

df_gpu = cudf.from_pandas(df_cpu)
print(df_gpu.head())
print("Filas:", len(df_gpu), "| Backend:", type(df_gpu).__module__)


In [ ]:
# 2) EDA comparable en CPU y GPU
start = time.perf_counter()
cpu_summary = df_cpu[FEATURES].describe()
t_cpu_eda = time.perf_counter() - start

# Warmup GPU fuera de la medición.
_ = df_gpu[FEATURES].mean()
cp.cuda.Stream.null.synchronize()

start = time.perf_counter()
gpu_summary = df_gpu[FEATURES].describe()
cp.cuda.Stream.null.synchronize()  # Espera a que la GPU termine antes de cerrar el tiempo.
t_gpu_eda = time.perf_counter() - start

display(gpu_summary)
print("Nulos por columna (GPU):")
print(df_gpu.isna().sum())
print(f"EDA CPU: {t_cpu_eda:.4f} s | EDA GPU: {t_gpu_eda:.4f} s")


In [ ]:
# 3) Mismo split para los dos backends
from sklearn.model_selection import train_test_split

indices = np.arange(len(df_cpu))
train_idx, test_idx = train_test_split(
    indices,
    test_size=0.2,
    random_state=42,
    stratify=df_cpu[TARGET],
)

X_train_cpu = df_cpu.iloc[train_idx][FEATURES]
X_test_cpu = df_cpu.iloc[test_idx][FEATURES]
y_train_cpu = df_cpu.iloc[train_idx][TARGET]
y_test_cpu = df_cpu.iloc[test_idx][TARGET]

X_train_gpu = df_gpu.iloc[train_idx][FEATURES].astype("float32")
X_test_gpu = df_gpu.iloc[test_idx][FEATURES].astype("float32")
y_train_gpu = df_gpu.iloc[train_idx][TARGET].astype("int32")

print("Train/test:", len(train_idx), len(test_idx))


In [ ]:
# 4) Benchmark obligatorio: sklearn (CPU) frente a cuML (GPU)
from sklearn.linear_model import LogisticRegression as SkLogReg
from sklearn.metrics import accuracy_score
from cuml.linear_model import LogisticRegression as CuLogReg

# Warmup de ambos backends, excluido de la medición.
SkLogReg(max_iter=50).fit(X_train_cpu.iloc[:2_048], y_train_cpu.iloc[:2_048])
CuLogReg(max_iter=50).fit(X_train_gpu.iloc[:2_048], y_train_gpu.iloc[:2_048])
cp.cuda.Stream.null.synchronize()

start = time.perf_counter()
model_cpu = SkLogReg(max_iter=200).fit(X_train_cpu, y_train_cpu)
t_cpu_fit = time.perf_counter() - start

start = time.perf_counter()
model_gpu = CuLogReg(max_iter=200).fit(X_train_gpu, y_train_gpu)
cp.cuda.Stream.null.synchronize()  # Imprescindible antes de detener el cronómetro.
t_gpu_fit = time.perf_counter() - start

pred_cpu = model_cpu.predict(X_test_cpu)
pred_gpu = model_gpu.predict(X_test_gpu)
cp.cuda.Stream.null.synchronize()

acc_cpu = accuracy_score(y_test_cpu, pred_cpu)
acc_gpu = accuracy_score(y_test_cpu, pred_gpu.to_numpy())

metrics = {
    "accuracy_cpu": float(acc_cpu),
    "accuracy_gpu": float(acc_gpu),
    "tiempo_cpu_eda_s": t_cpu_eda,
    "tiempo_gpu_eda_s": t_gpu_eda,
    "tiempo_cpu_fit_s": t_cpu_fit,
    "tiempo_gpu_fit_s": t_gpu_fit,
    "speedup_fit_cpu_sobre_gpu": t_cpu_fit / t_gpu_fit,
    "gpu": str(device_name),
    "cudf_version": cudf.__version__,
    "cuml_version": cuml.__version__,
}
print(json.dumps(metrics, indent=2, ensure_ascii=False))


In [ ]:
# 5) Exportación de los artefactos requeridos
import joblib

artifacts = Path("artifacts")
artifacts.mkdir(exist_ok=True)
joblib.dump(model_gpu, artifacts / "model.joblib")
(artifacts / "metrics.json").write_text(
    json.dumps(metrics, indent=2, ensure_ascii=False),
    encoding="utf-8",
)
df_gpu.to_pandas().to_csv(artifacts / "clean.csv", index=False)
print("Artefactos guardados en", artifacts.resolve())


## Análisis obligatorio

Explica en la entrega:

- qué GPU y versiones utilizaste;
- por qué excluiste el warmup y sincronizaste antes de detener el cronómetro;
- qué ruta fue más rápida y por qué;
- cómo influyen el tamaño de los datos y las transferencias CPU↔GPU;
- qué cambiarías antes de llevar este flujo a producción.

Después integra los artefactos en el dashboard y el pipeline de las prácticas 2 y 3.